# 03 — Preparo do HMDA por ano

Separamos os pedidos dos oito snapshots de 2018 a 2025. Para cada ano, conservamos compras de residência principal que receberam aprovação ou recusa e passaram na validação dos 18 campos. O ano de 2017 tem outro formato e exige um recorte de variáveis diferente.

O notebook lê cada CSV diretamente do ZIP em blocos e gera um Parquet por ano com a partição de cada pedido. A separação é estratificada por ano e decisão, em proporções aproximadas de 70% para treino, 20% para validação e 10% para teste. Confira as contagens antes de abrir o notebook 04.

In [1]:
from pathlib import Path
import sys
import pandas as pd

try:
    from google.colab import drive
except ImportError:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
else:
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/hmda-credit-fairness')

sys.path.insert(0, str(ROOT / 'scripts'))
from hmda_multiano import YEARS, prepare_year

RAW = ROOT / 'data/raw'
PROCESSED = ROOT / 'data/processed/hmda_multiano'
archives = {year: RAW / f'{year}_public_lar_csv.zip' for year in YEARS}
missing = [str(path) for path in archives.values() if not path.is_file()]
if missing:
    raise FileNotFoundError('Arquivos oficiais ausentes: ' + ', '.join(missing))

In [2]:
audits, partitions, invalid = [], [], []
for year, archive in archives.items():
    print(f'Validando {year}', flush=True)
    audit, parts, fields = prepare_year(archive, PROCESSED / f'hmda_{year}_valid18.parquet', year)
    audits.append(audit)
    partitions.extend(parts)
    invalid.extend(fields)
    print(f'{year}: {audit["validos"]:,} pedidos válidos', flush=True)

for name, records in [('auditoria_por_ano', audits), ('particoes_por_ano', partitions),
                      ('invalidos_por_campo_e_ano', invalid)]:
    table = pd.DataFrame(records)
    table.to_csv(PROCESSED / f'{name}.csv', index=False)
    if name != 'invalidos_por_campo_e_ano':
        display(table)
print(f'Parquets e relatórios prontos em {PROCESSED}')

Validando 2018
2018: 3,655,642 pedidos válidos
Validando 2019
2019: 3,724,416 pedidos válidos
Validando 2020
2020: 4,075,811 pedidos válidos
Validando 2021
2021: 4,306,256 pedidos válidos
Validando 2022
2022: 3,568,949 pedidos válidos
Validando 2023
2023: 2,884,982 pedidos válidos
Validando 2024
2024: 2,910,642 pedidos válidos
Validando 2025
2025: 2,942,508 pedidos válidos


,ano,lidos,elegiveis,validos,excluidos
0,2018,15119651,4605550,3655642,949908
1,2019,17545457,4743307,3724416,1018891
2,2020,25551868,5204454,4075811,1128643
3,2021,26124552,5480901,4306256,1174645
4,2022,16080210,4611056,3568949,1042107
5,2023,11483889,3772167,2884982,887185
6,2024,12229298,3794116,2910642,883474
7,2025,13543606,3828884,2942508,886376


,ano,particao,n,aprovados,recusados
0,2018,ajuste,2558946,2293963,264983
1,2018,validacao,731141,655425,75716
2,2018,teste,365555,327703,37852
3,2019,ajuste,2607091,2359519,247572
4,2019,validacao,744888,674149,70739
5,2019,teste,372437,337073,35364
6,2020,ajuste,2853064,2569747,283317
7,2020,validacao,815168,734217,80951
8,2020,teste,407579,367104,40475
9,2021,ajuste,3014377,2726355,288022


Parquets e relatórios prontos em c:\Users\cg3034593\Documents\artigo\data\processed\hmda_multiano
